# MNIST Training Notebook

This notebook trains the standard PyTorch MNIST convolutional network and displays live Caliper views while the model trains. The views show the training loss and accuracy curves, the 32 first-layer filters, the first-layer activations for one fixed test digit, a 3-D point cloud of the hidden layer colored by digit class, and a histogram of the `fc1` weights.

The model, the Adadelta optimizer, the learning rate schedule, and the pixel normalization follow the official PyTorch MNIST example (`pytorch/examples/mnist/main.py`). The notebook needs `caliper[notebook]`, `torch`, `torchvision`, and `ipywidgets`.

## Setup and Dataset

The first cell imports PyTorch, `torchvision`, and Caliper. It selects Apple Silicon MPS when MPS is available, else CUDA or the CPU. Then it downloads MNIST into the cache folder `~/.cache/caliper/mnist` (or `$CALIPER_MNIST_DATA_DIR`) and builds a fixed batch of 512 test digits. Item 0 of the batch is the fixed digit for the activation view, and a fixed orthonormal matrix projects the 128-unit hidden layer into three dimensions.

In [ ]:
import os
from pathlib import Path
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import caliper

# Standard PyTorch MNIST normalization constants (pytorch/examples/mnist/main.py).
MNIST_MEAN = (0.1307,)
MNIST_STD = (0.3081,)
DATA_DIR = Path(
    os.environ.get("CALIPER_MNIST_DATA_DIR", str(Path.home() / ".cache" / "caliper" / "mnist"))
)
EPOCHS = int(os.environ.get("CALIPER_MNIST_EPOCHS", "14"))
_steps_env = os.environ.get("CALIPER_NOTEBOOK_STEPS", "").strip()
MAX_STEPS = int(_steps_env) if _steps_env else None
BATCH_SIZE = 64
TEST_BATCH_SIZE = 1000
PROBE_EVERY = 10
PROBE_SIZE = 512

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"device {device}, torch {torch.__version__}")

torch.manual_seed(1)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MNIST_MEAN, MNIST_STD),
])
DATA_DIR.mkdir(parents=True, exist_ok=True)
train_set = datasets.MNIST(root=str(DATA_DIR), train=True, download=True, transform=transform)
test_set = datasets.MNIST(root=str(DATA_DIR), train=False, download=True, transform=transform)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_set, batch_size=TEST_BATCH_SIZE, shuffle=False)

probe_images, probe_labels = zip(*(test_set[i] for i in range(min(PROBE_SIZE, len(test_set)))))
probe_x = torch.stack(probe_images).to(device)
probe_y = torch.tensor(probe_labels, dtype=torch.long, device=device)
proj, _ = torch.linalg.qr(torch.randn(128, 3, generator=torch.Generator().manual_seed(0)))
proj = proj.to(device)

## Model and Watch

`Net` is the two-convolution, two-linear network from `pytorch/examples/mnist/main.py`. `caliper.watch(model, optimizer, activations=False, groups=False)` starts the local Caliper stream server, publishes the `loss` and `lr` series automatically at each optimizer step, and registers the interactive `lr` control.

In [ ]:
# Standard PyTorch MNIST CNN from pytorch/examples/mnist/main.py:
# https://github.com/pytorch/examples/blob/main/mnist/main.py
class Net(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)


model = Net().to(device)
optimizer = torch.optim.Adadelta(model.parameters(), lr=1.0)
scheduler = StepLR(optimizer, step_size=1, gamma=0.7)
watch = caliper.watch(model, optimizer, activations=False, groups=False)

## Live Views

Each `caliper.widget` connects directly to the stream server and continues to update while the training cell runs. `caliper.controls()` shows the run state, the Pause, Resume, and Step buttons, and the learning rate slider.

In [ ]:
from ipywidgets import HBox, VBox

loss_w = caliper.widget("loss", "series", height=220)
acc_w = caliper.widget("accuracy", "series", height=220)
filters_w = caliper.widget("conv1.filters", "heatmap", height=280)
act_w = caliper.widget("conv1.act", "heatmap", height=280)
emb_w = caliper.widget("embedding", "points", height=320)
hist_w = caliper.widget("fc1.weight.hist", "histogram", height=320)

for w in (loss_w, acc_w, filters_w, act_w, emb_w, hist_w):
    w.layout.flex = "1 1 0"

VBox([
    caliper.controls(),
    HBox([loss_w, acc_w]),
    HBox([filters_w, act_w]),
    HBox([emb_w, hist_w]),
])

## Training Loop

The training loop matches `examples/mnist/train.py`. `caliper.watch` publishes `loss` and `lr` at each optimizer step. The loop passes tensors on the compute device directly to `caliper.series`, `caliper.show`, `caliper.points`, and `caliper.histogram`, so training does not wait for GPU readback. Every 10 steps, a `torch.no_grad()` pass on the fixed test batch updates the `conv1.act` mosaic for test digit 0 and the 10-class `embedding` point cloud.

In [ ]:
probe_acts = {}
capturing = False


def _save_conv1(_mod, _inp, out):
    if capturing:
        probe_acts["conv1"] = F.relu(out[0].detach())


def _save_hidden(_mod, _inp, out):
    if capturing:
        probe_acts["hidden"] = F.relu(out.detach())


def publish_probe():
    global capturing
    was_training = model.training
    model.eval()
    capturing = True
    try:
        with torch.no_grad():
            model(probe_x)
    finally:
        capturing = False
        if was_training:
            model.train()
    caliper.show(probe_acts["conv1"], "conv1.act")
    caliper.points(probe_acts["hidden"] @ proj, probe_y, "embedding", classes=10)


def evaluate():
    was_training = model.training
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            out = model(data)
            total_loss += F.nll_loss(out, target, reduction="sum").item()
            correct += out.argmax(dim=1).eq(target).sum().item()
            total += target.size(0)
    if was_training:
        model.train()
    return total_loss / max(total, 1), correct / max(total, 1)


h_conv1 = model.conv1.register_forward_hook(_save_conv1)
h_fc1 = model.fc1.register_forward_hook(_save_hidden)
step = 0
t0 = time.perf_counter()
model.train()
try:
    for epoch in range(1, EPOCHS + 1):
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = F.nll_loss(output, target)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            step += 1
            accuracy = (output.argmax(dim=1) == target).float().mean()
            caliper.series(accuracy, "accuracy")
            caliper.show(
                model.conv1.weight.squeeze(1),
                "conv1.filters",
                cmap="rdbu",
                dims=("filter", "height", "width"),
                data_view={"tile": "filter"},
            )
            caliper.histogram(model.fc1.weight, "fc1.weight.hist")
            if step == 1 or step % PROBE_EVERY == 0 or (MAX_STEPS is not None and step == MAX_STEPS):
                publish_probe()

            if MAX_STEPS is not None and step >= MAX_STEPS:
                break

        train_loss = loss.item()
        if MAX_STEPS is not None and step >= MAX_STEPS:
            break
        scheduler.step()
        test_loss, test_acc = evaluate()
        print(
            f"epoch {epoch}/{EPOCHS}  step {step}  train_loss {train_loss:.4f}  "
            f"test_loss {test_loss:.4f}  test_acc {100.0 * test_acc:.2f}%"
        )
finally:
    h_conv1.remove()
    h_fc1.remove()

if device.type == "mps":
    torch.mps.synchronize()
elapsed = time.perf_counter() - t0
test_loss, test_acc = evaluate()
print(
    f"done  steps={step}  elapsed={elapsed:.2f}s  train_loss={train_loss:.4f}  "
    f"test_loss={test_loss:.4f}  test_acc={100.0 * test_acc:.2f}%"
)

## Cleanup

Remove the watch hooks and close the stream server after inspecting the views.

In [ ]:
watch.remove()
caliper.close()